# 01 · Adquisición de datos

**Objetivo.** Obtener los datos de eventos de fútbol y construir un dataset a nivel de disparo (una fila por tiro), de forma automática y reproducible.

**Fuente.** [StatsBomb Open Data](https://github.com/statsbomb/open-data) vía `statsbombpy`. El módulo `src.data_loader` descarga competiciones, partidos y eventos, filtra los disparos y extrae los *freeze frames* (posiciones de jugadores en el instante del tiro), base de las variables defensivas.

**Robustez.** Si no hay conexión o `statsbombpy` no está instalado, el loader genera un dataset **sintético físicamente realista** con idéntica estructura, de modo que el pipeline es siempre ejecutable. El generador define la probabilidad de gol mediante un modelo logístico de distancia y ángulo, por lo que el *xG verdadero* es conocido y permite validar la calibración.

In [1]:
# Bootstrap: hace importable el paquete `xg` desde notebooks/ (src layout)
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == 'notebooks') else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from xg import config, features, models, analysis, visualization
from xg import data as data_loader
from xg.models import evaluation, statistics
pd.set_option('display.float_format', lambda v: f'{v:0.4f}')
config.ensure_dirs()
np.random.seed(config.RANDOM_STATE)
print('Proyecto:', config.PROJECT_ROOT.name)

Proyecto: TFM_xG_MachineLearning


## 1.1 Carga unificada de disparos
`build_shots_dataframe` intenta StatsBomb y cae al sintético si es necesario. Para usar datos reales explícitamente: `source='statsbomb'`.

In [2]:
raw, source = data_loader.build_shots_dataframe(source='auto')
print('Fuente utilizada:', source)
print('Número de disparos:', len(raw))
raw.head()

Fuente utilizada: statsbomb
Número de disparos: 5829


,match_id,competition,period,event_index,team,player,minute,second,x,y,...,technique,play_pattern,under_pressure,is_first_time,assisted,n_defenders_in_cone,n_defenders_within_3m,distance_to_nearest_defender,gk_distance_to_goal,gk_distance_to_shot
0,7585,FIFA World Cup 2018,1,240,England,Ashley Young,5,22,115.0000,18.0000,...,Normal,From Free Kick,False,False,False,2.0000,0.0000,10.0499,1.0000,23.5372
1,7585,FIFA World Cup 2018,1,271,England,Raheem Sterling,7,6,112.0000,54.0000,...,Normal,From Free Kick,False,False,True,1.0000,2.0000,1.0000,4.0000,12.8062
2,7585,FIFA World Cup 2018,1,440,England,Raheem Sterling,12,40,98.0000,37.0000,...,Normal,From Counter,True,False,True,2.0000,2.0000,1.4142,1.0000,22.3607
3,7585,FIFA World Cup 2018,1,549,England,Harry Kane,15,28,119.0000,36.0000,...,Normal,From Throw In,False,False,True,1.0000,1.0000,2.8284,3.0000,7.0711
4,7585,FIFA World Cup 2018,1,816,Colombia,Juan Guillermo Cuadrado Bello,21,53,97.0000,56.0000,...,Normal,Regular Play,False,False,True,1.0000,1.0000,2.0000,1.0000,27.2029


## 1.2 Inspección de calidad de datos

In [3]:
print('Dimensiones:', raw.shape)
print('\nTipos de dato:')
print(raw.dtypes)
print('\nValores nulos por columna:')
print(raw.isna().sum())

Dimensiones: (5829, 24)

Tipos de dato:
match_id                          int64
competition                         str
period                            int64
event_index                       int64
team                                str
player                              str
minute                            int64
second                            int64
x                               float64
y                               float64
is_goal                           int64
statsbomb_xg                    float64
body_part                           str
shot_type                           str
technique                           str
play_pattern                        str
under_pressure                     bool
is_first_time                      bool
assisted                           bool
n_defenders_in_cone             float64
n_defenders_within_3m           float64
distance_to_nearest_defender    float64
gk_distance_to_goal             float64
gk_distance_to_shot             float64


In [4]:
print('Tasa de gol global: %.4f' % raw['is_goal'].mean())
print('\nDisparos por tipo:')
print(raw['shot_type'].value_counts())
print('\nDisparos por parte del cuerpo:')
print(raw['body_part'].value_counts())

Tasa de gol global: 0.1131

Disparos por tipo:
shot_type
Open Play    5388
Penalty       223
Free Kick     212
Corner          6
Name: count, dtype: int64

Disparos por parte del cuerpo:
body_part
Right Foot    3034
Left Foot     1746
Head          1011
Other           38
Name: count, dtype: int64


## 1.3 Persistencia
Guardamos el dataset crudo en `data/processed/` (CSV + binario).

In [5]:
print('Datos crudos cargados. El guardado oficial lo realiza run_pipeline.py.')

Datos crudos cargados. El guardado oficial lo realiza run_pipeline.py.


**Conclusión.** Disponemos de un dataset de disparos limpio y reproducible, listo para el análisis exploratorio (Notebook 02).